In [1]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def guess_time_voltage_columns(df):
    cols = [c.strip() for c in df.columns]
    df = df.copy()
    df.columns = cols
    if "in s" in cols and "C1 in V" in cols:
        return "in s", "C1 in V", df
    # fallback: first two numeric columns
    numeric_cols = []
    for c in cols:
        try:
            pd.to_numeric(df[c].iloc[:50], errors="raise")
            numeric_cols.append(c)
        except Exception:
            pass
    if len(numeric_cols) >= 2:
        return numeric_cols[0], numeric_cols[1], df
    raise ValueError("Could not infer time/voltage columns")

def first_sustained(mask, N, frac):
    N = max(2, int(N))
    counts = np.convolve(mask.astype(int), np.ones(N, dtype=int), mode="valid")
    hits = np.where(counts >= frac * N)[0]
    return None if len(hits) == 0 else int(hits[0])

def compute_start_end(t, v, pre_plateau_us=1.0, post_plateau_us=1.0,
                      k=5, start_sustain_ns=80, end_sustain_us=0.8,
                      frac_start=0.95, frac_end=0.98):
    order = np.argsort(t)
    t, v = t[order], v[order]
    dt = float(np.median(np.diff(t)))
    peak_idx = int(np.argmax(v))
    tmin, tmax = float(t.min()), float(t.max())

    pre_mask  = t < (tmin + pre_plateau_us * 1e-6)
    post_mask = t > (tmax - post_plateau_us * 1e-6)

    base_pre  = float(np.median(v[pre_mask])) if pre_mask.sum() > 10 else float(np.median(v))
    sig_pre   = float(np.std(v[pre_mask], ddof=1)) if pre_mask.sum() > 10 else float(np.std(v, ddof=1))

    base_post = float(np.median(v[post_mask])) if post_mask.sum() > 10 else float(np.median(v))
    sig_post  = float(np.std(v[post_mask], ddof=1)) if post_mask.sum() > 10 else float(np.std(v, ddof=1))

    Nstart = int(round((start_sustain_ns * 1e-9) / dt))
    Nend   = int(round((end_sustain_us  * 1e-6) / dt))

    thr_start = base_pre + k * sig_pre
    start_idx = first_sustained(v > thr_start, Nstart, frac_start)

    low, high = base_post - k * sig_post, base_post + k * sig_post
    within_post = (v >= low) & (v <= high)
    within_post[:peak_idx] = False
    end_idx = first_sustained(within_post, Nend, frac_end)

    ok = (start_idx is not None) and (end_idx is not None)
    return ok, (t[start_idx] if ok else np.nan), (t[end_idx] if ok else np.nan), base_pre, base_post

# -------- Batch over files --------
files = sorted(glob.glob("D*.CSV") + glob.glob("D*.csv"))
if not files:
    files = sorted(glob.glob("/mnt/data/D*.CSV") + glob.glob("/mnt/data/D*.csv"))
if not files:
    raise FileNotFoundError("No D*.CSV files found.")

out_dir = "peak_plots"
os.makedirs(out_dir, exist_ok=True)

rows = []

# One plot per file
for f in files:
    df = pd.read_csv(f)
    t_col, v_col, df = guess_time_voltage_columns(df)
    t = pd.to_numeric(df[t_col], errors="coerce").to_numpy()
    v = pd.to_numeric(df[v_col], errors="coerce").to_numpy()
    m = np.isfinite(t) & np.isfinite(v)
    t, v = t[m], v[m]

    ok, t_start, t_end, base_pre, base_post = compute_start_end(t, v)

    plt.figure()
    plt.plot(t*1e6, v, linewidth=1)
    plt.xlabel("Time (µs)")
    plt.ylabel("Voltage (V)")
    plt.title(os.path.basename(f))
    plt.grid(True)

    if ok:
        plt.axvline(t_start*1e6, linestyle="--")
        plt.axvline(t_end*1e6, linestyle="--")
        plt.axhline(base_pre, linestyle=":")
        plt.axhline(base_post, linestyle=":")

    plt.tight_layout()
    png_path = os.path.join(out_dir, os.path.splitext(os.path.basename(f))[0] + ".png")
    plt.savefig(png_path, dpi=200)
    plt.close()

    rows.append({
        "file": os.path.basename(f),
        "ok": ok,
        "start_us": t_start*1e6 if ok else np.nan,
        "end_us": t_end*1e6 if ok else np.nan,
        "duration_us": (t_end - t_start)*1e6 if ok else np.nan,
        "plot_png": png_path
    })

# Overlay plot
plt.figure()
for f in files:
    df = pd.read_csv(f)
    t_col, v_col, df = guess_time_voltage_columns(df)
    t = pd.to_numeric(df[t_col], errors="coerce").to_numpy()
    v = pd.to_numeric(df[v_col], errors="coerce").to_numpy()
    m = np.isfinite(t) & np.isfinite(v)
    t, v = t[m], v[m]
    order = np.argsort(t)
    plt.plot(t[order]*1e6, v[order], linewidth=1, label=os.path.splitext(os.path.basename(f))[0])

plt.xlabel("Time (µs)")
plt.ylabel("Voltage (V)")
plt.title("Overlay: D* traces")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "overlay.png"), dpi=200)
plt.close()

summary = pd.DataFrame(rows).sort_values("file")
summary.to_csv(os.path.join(out_dir, "summary.csv"), index=False)
print(summary[["file","ok","duration_us","plot_png"]])


     file    ok  duration_us           plot_png
0  D1.CSV  True       3.2364  peak_plots/D1.png
1  D2.CSV  True       3.0560  peak_plots/D2.png
2  D3.CSV  True       3.0352  peak_plots/D3.png
3  D4.CSV  True       2.9928  peak_plots/D4.png
4  D5.CSV  True       3.2712  peak_plots/D5.png


3.12 +-0.06